### MySQL 연동

In [2]:
import numpy as np
import pandas as pd
import pymysql
from sqlalchemy import create_engine

#### cp949 형식의 csv 파일을 읽어 utf-8형식의 csv파일로 저장 

In [4]:
df = pd.read_csv('bidResCategorizedCP949_armyClassified.csv',encoding='cp949', low_memory=False)
df.head(2)

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,업무구분,발주기관코드,발주기관,...,낙찰자(영문상호),낙찰자(대표자),낙찰자(연락처),낙찰자(연락처).1,낙찰자(주소),낙찰률,낙찰금액,카테고리,pick_keywords,군 분류
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,물품,SFL,제3283부대,...,Jung woo military,이정구,222730487,02-2273-0487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,"23,689,950",2종,명찰,육군
1,2,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,물품,LFC,육군훈련소,...,LNF,이용환,437318109,043-731-8109,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,"5,810",1종,도시락,미분류


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74732 entries, 0 to 74731
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   bidID          74732 non-null  int64  
 1   참가수            74732 non-null  int64  
 2   공고번호           74732 non-null  object 
 3   공고차수           74732 non-null  int64  
 4   판단번호           74732 non-null  object 
 5   항목번호           74732 non-null  object 
 6   입찰명            74732 non-null  object 
 7   업무구분           74732 non-null  object 
 8   발주기관코드         74732 non-null  object 
 9   발주기관           74732 non-null  object 
 10  개찰일시           74732 non-null  object 
 11  낙찰자(업체코드)      74732 non-null  object 
 12  낙찰자(상호)        74732 non-null  object 
 13  낙찰자(영문상호)      53075 non-null  object 
 14  낙찰자(대표자)       74732 non-null  object 
 15  낙찰자(연락처)       74732 non-null  object 
 16  낙찰자(연락처).1     74732 non-null  object 
 17  낙찰자(주소)        74732 non-null  object 
 18  낙찰률   

- 결측치 확인

In [17]:
df.isna().sum()

bidID                0
참가수                  0
공고번호                 0
공고차수                 0
판단번호                 0
항목번호                 0
입찰명                  0
업무구분                 0
발주기관코드               0
발주기관                 0
개찰일시                 0
낙찰업체코드               0
낙찰업체상호               0
영문상호             21657
대표자                  0
연락처                  0
연락처2                 0
업체주소                 0
낙찰률                133
낙찰금액                 2
카테고리                 0
pick_keywords    20014
군 분류                 0
dtype: int64

#### 데이터 전처리

In [7]:
cols = df.columns
print(cols)

Index(['bidID', '참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분', '발주기관코드',
       '발주기관', '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)',
       '낙찰자(연락처)', '낙찰자(연락처).1', '낙찰자(주소)', '낙찰률', ' 낙찰금액 ', '카테고리',
       'pick_keywords', '군 분류'],
      dtype='object')


- 컬럼명 변경

In [10]:
columns=['bidID', '참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분',
       '발주기관코드', '발주기관', '개찰일시', '낙찰업체코드', '낙찰업체상호', '영문상호',
       '대표자', '연락처', '연락처2', '업체주소', '낙찰률', '낙찰금액','카테고리', 'pick_keywords', '군 분류']

In [11]:
df.columns = columns

- 각 행 구분을 위한 ID 컬럼의 값을 1부터 시작하도록

In [8]:
# df['ID'] =df['ID'] + 1

In [9]:
df['bidID'].describe()

count    74732.000000
mean     37366.500000
std      21573.414496
min          1.000000
25%      18683.750000
50%      37366.500000
75%      56049.250000
max      74732.000000
Name: bidID, dtype: float64

- 데이터 포맷 변경

In [12]:
df['개찰일시'] = pd.to_datetime(df['개찰일시']) 

In [13]:
col = '낙찰금액'
# 숫자/마이너스만 허용
s = df[col].str.strip().str.replace(r'[^\d\-]', '', regex=True)

# 숫자(float64 또는 int64)로 변환
df['낙찰금액'] = pd.to_numeric(s, errors='coerce')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74732 entries, 0 to 74731
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   bidID          74732 non-null  int64         
 1   참가수            74732 non-null  int64         
 2   공고번호           74732 non-null  object        
 3   공고차수           74732 non-null  int64         
 4   판단번호           74732 non-null  object        
 5   항목번호           74732 non-null  object        
 6   입찰명            74732 non-null  object        
 7   업무구분           74732 non-null  object        
 8   발주기관코드         74732 non-null  object        
 9   발주기관           74732 non-null  object        
 10  개찰일시           74732 non-null  datetime64[ns]
 11  낙찰업체코드         74732 non-null  object        
 12  낙찰업체상호         74732 non-null  object        
 13  영문상호           53075 non-null  object        
 14  대표자            74732 non-null  object        
 15  연락처            7473

In [14]:
df.head()

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,업무구분,발주기관코드,발주기관,...,영문상호,대표자,연락처,연락처2,업체주소,낙찰률,낙찰금액,카테고리,pick_keywords,군 분류
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,물품,SFL,제3283부대,...,Jung woo military,이정구,222730487,02-2273-0487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,2.368995e+07,2종,명찰,육군
1,2,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,물품,LFC,육군훈련소,...,LNF,이용환,437318109,043-731-8109,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,5.810000e+03,1종,도시락,미분류
2,3,22,LGT0034,1,48136,***,0부대 중대시설 분전반 제조설치(16-312),물품,LGT,제2307부대,...,seorak tech,김화성,334610399,033-461-0399,강원특별자치도 인제군 북면 원통로74번길10-5(원통농공단지),88.0408,2.909872e+07,4종,제조설치,육군
3,4,22,UMM0839,1,48049,***,13-본-국대-01 00학교 가구 및 비품 제조설치,물품,UMM,국군재정관리단,...,"DS NAIKI CO.,LTD",김준홍,27342125,02-734-2125,"서울특별시 구로구 구로중앙로134, 4층(구로동, 리치몰)",88.2297,2.165042e+09,2종,가구,기타
4,5,9,LGT0035,1,48138,***,00부대 중대시설 조명기구 제조납품(16-314),물품,LGT,제2307부대,...,"HANSEO Co.,LTD",이명옥,337462110,033-746-2110,강원특별자치도 원주시 태장공단길47-0 (태장동),88.3713,3.242441e+07,4종,조명기구,육군


In [25]:
df2.to_csv('BID.csv',encoding='cp949', index=False)

In [16]:
df2 = pd.read_csv('BID.csv', encoding='cp949')
df2.head(2)

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,업무구분,발주기관코드,발주기관,...,영문상호,대표자,연락처,연락처2,업체주소,낙찰률,낙찰금액,카테고리,pick_keywords,군 분류
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,물품,SFL,제3283부대,...,Jung woo military,이정구,222730487,02-2273-0487,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,23689950.0,2종,명찰,육군
1,2,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,물품,LFC,육군훈련소,...,LNF,이용환,437318109,043-731-8109,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,5810.0,1종,도시락,미분류


In [49]:
df[df.낙찰금액.isna()][['bidID', '입찰명','개찰일시', '낙찰률', '낙찰금액']]

,bidID,입찰명,개찰일시,낙찰률,낙찰금액
53311,53312,검독수리-B B-II 1~4번함 건조,2022-10-24 14:00:00,NaN,NaN
60032,60033,AI기반 전술차량용 지능형 RCWS,2023-12-12 10:00:00,0.0,NaN


In [ ]:
df = df[~df['bidID'].isin([53312, 60033])]

In [48]:
df[df.bidID == 53312]['입찰명']

53311    검독수리-B B-II 1~4번함 건조
Name: 입찰명, dtype: object

In [28]:
df3 = df2.drop(columns = ['영문상호', '연락처', '연락처2', '업무구분'])
df3.columns

Index(['bidID', '참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '발주기관코드', '발주기관',
       '개찰일시', '낙찰업체코드', '낙찰업체상호', '대표자', '업체주소', '낙찰률', '낙찰금액', '카테고리',
       'pick_keywords', '군 분류'],
      dtype='object')

In [38]:
df3.to_csv('BID_fianl.csv',encoding='cp949', index=False)

In [30]:
# 조건별로 갱신
df3.loc[df["발주기관"].str.contains("육군", na=False), "군 분류"] = "육군"
df3.loc[df["발주기관"].str.contains("해군", na=False), "군 분류"] = "해군"
df3.loc[df["발주기관"].str.contains("공군", na=False), "군 분류"] = "공군"

In [37]:
df3[df3['군 분류'] == '미분류'][['군 분류']]

,군 분류


In [35]:
df3.head(15)

,bidID,참가수,공고번호,공고차수,판단번호,항목번호,입찰명,발주기관코드,발주기관,개찰일시,낙찰업체코드,낙찰업체상호,대표자,업체주소,낙찰률,낙찰금액,카테고리,pick_keywords,군 분류
0,1,81,SFL0160,1,47669,***,임시명찰 등 2품목 구매,SFL,제3283부대,2016-12-30 12:30:00,H84F8,주식회사 전우밀리터리,이정구,"경기도 고양시 덕양구 통일로140, 1층 에이143호, 2층 에이261호,지하1층 ...",88.0062,2.368995e+07,2종,명찰,육군
1,2,1,LFC0043,1,45002,1,장거리 이동 신병 도시락 제조납품,LFC,육군훈련소,2016-12-30 10:00:00,GA6E7,주식회사 엘엔에프 농업회사법인,이용환,충청북도 옥천군 옥천읍 삼청리877-1,96.8333,5.810000e+03,1종,도시락,육군
2,3,22,LGT0034,1,48136,***,0부대 중대시설 분전반 제조설치(16-312),LGT,제2307부대,2016-12-29 11:00:00,F91A4,설악테크,김화성,강원특별자치도 인제군 북면 원통로74번길10-5(원통농공단지),88.0408,2.909872e+07,4종,제조설치,육군
3,4,22,UMM0839,1,48049,***,13-본-국대-01 00학교 가구 및 비품 제조설치,UMM,국군재정관리단,2016-12-29 11:00:00,B30CC,주식회사 디에스나이키,김준홍,"서울특별시 구로구 구로중앙로134, 4층(구로동, 리치몰)",88.2297,2.165042e+09,2종,가구,기타
4,5,9,LGT0035,1,48138,***,00부대 중대시설 조명기구 제조납품(16-314),LGT,제2307부대,2016-12-29 11:00:00,C959B,주식회사 한서,이명옥,강원특별자치도 원주시 태장공단길47-0 (태장동),88.3713,3.242441e+07,4종,조명기구,육군
5,6,4,HCL0100,1,48057,***,2017년 난방연료,HCL,공군사관학교,2016-12-29 10:30:00,GAA78,주식회사거성에너지,박정봉,충청북도 충주시 서흥1길22-0 (목행동),90.4952,1.497000e+08,3종,연료,공군
6,7,8,LNF0041,1,49073,***,00부대 체육시설 비품구매(고소작업대 등 9종),LNF,제1보병사단,2016-12-29 10:30:00,H0239,주연ICT,이주연,경기도 파주시 중앙로279-0 (아동동),88.1355,2.238018e+07,"2종, 7종","고소작업대, 체육",육군
7,8,2,HCM0002,1,19,1,17년 액화석유가스(LPG) 구매,HCM,공군교육사령부,2016-12-28 11:00:00,ECA82,주식회사 가스마켓코리아,박영숙,경상남도 창원시 의창구 동읍 동읍로1033-0,88.1977,1.889000e+03,3종,가스,공군
8,9,3,HCL0097,2,46235,***,군수처 취사연료,HCL,공군사관학교,2016-12-28 10:30:00,GE2D7,SK가스충청충전소,홍금성,충청북도 청주시 흥덕구 가로수로1086번길20-0 (비하동),88.0299,7.351997e+07,3종,연료,공군
9,10,2,SFC0229,1,47835,***,우갈비 등 15품목 (덕산스포텔),SFC,제9691부대,2016-12-28 10:30:00,G75EF,충성축산,석관희,경기도 화성시 정남면 만년로440-0,92.1355,5.722720e+07,1종,갈비,해병


In [47]:
df3[df3['군 분류'] == '육군'][['군 분류', '발주기관']]

,군 분류,발주기관
0,육군,제3283부대
1,육군,육군훈련소
2,육군,제2307부대
4,육군,제2307부대
6,육군,제1보병사단
...,...,...
74718,육군,육군훈련소
74719,육군,육군훈련소
74726,육군,제 2167부대
74730,육군,제3707부대


---